In [1]:
import sys
from pathlib import Path

import pandas as pd

print("Python:", sys.executable)
print("Working folder:", Path.cwd())

Python: c:\Users\clark\OneDrive\Documents\Python Scripts\movie-recommendation-system\.venv\Scripts\python.exe
Working folder: c:\Users\clark\OneDrive\Documents\Python Scripts\movie-recommendation-system\notebooks


In [ ]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data" / "raw" / "ml-latest-small"

movies = pd.read_csv(data_dir / "movies.csv")
ratings = pd.read_csv(data_dir / "ratings.csv")

In [3]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [5]:
print("Movies table shape:", movies.shape)
print("Ratings table shape:", ratings.shape)

print("Unique users:", ratings["userId"].nunique())
print("Unique rated movies:", ratings["movieId"].nunique())

Movies table shape: (9742, 3)
Ratings table shape: (100836, 4)
Unique users: 610
Unique rated movies: 9724


In [6]:
has_rating = movies["movieId"].isin(ratings["movieId"])

movies_without_ratings = movies.loc[~has_rating]

print("Movies without ratings:", len(movies_without_ratings))

movies_without_ratings.head()

Movies without ratings: 18


,movieId,title,genres
816,1076,"Innocents, The (1961)",Drama|Horror|Thriller
2211,2939,Niagara (1953),Drama|Thriller
2499,3338,For All Mankind (1989),Documentary
2587,3456,"Color of Paradise, The (Rang-e khoda) (1999)",Drama
3118,4194,I Know Where I'm Going! (1945),Drama|Romance|War


In [7]:
movies.isna().sum()

movieId    0
title      0
genres     0
dtype: int64

In [8]:
ratings.isna().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [9]:
print(
    "Duplicate movie IDs:",
    movies["movieId"].duplicated().sum()
)

print(
    "Duplicate rating rows:",
    ratings.duplicated().sum()
)

print(
    "Repeated user-movie pairs:",
    ratings.duplicated(subset=["userId", "movieId"]).sum()
)

Duplicate movie IDs: 0
Duplicate rating rows: 0
Repeated user-movie pairs: 0


In [10]:
print("Movies — missing values:")
print(movies.isna().sum())

print("Ratings — missing values:")
print(ratings.isna().sum())

Movies — missing values:
movieId    0
title      0
genres     0
dtype: int64
Ratings — missing values:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


In [11]:
rating_counts = ratings["rating"].value_counts().sort_index()

rating_counts

rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64

In [12]:
movie_ids_match = ratings["movieId"].isin(movies["movieId"])

print(
    "Ratings with no matching catalog movie:",
    (~movie_ids_match).sum()
)

Ratings with no matching catalog movie: 0


In [13]:
ratings_with_movies = ratings.merge(
    movies,
    on="movieId",
    how="left",
    validate="many_to_one"
)

ratings_with_movies.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [14]:
print("Rows before join:", len(ratings))
print("Rows after join:", len(ratings_with_movies))

Rows before join: 100836
Rows after join: 100836


In [15]:
user_1_ratings = ratings_with_movies.loc[
    ratings_with_movies["userId"] == 1,
    ["title", "rating", "genres"]
]

user_1_ratings.sort_values(
    "rating",
    ascending=False
).head(10)

,title,rating,genres
3,Seven (a.k.a. Se7en) (1995),5.0,Mystery|Thriller
4,"Usual Suspects, The (1995)",5.0,Crime|Mystery|Thriller
6,Bottle Rocket (1996),5.0,Adventure|Comedy|Crime|Romance
13,Dumb & Dumber (Dumb and Dumber) (1994),5.0,Adventure|Comedy
11,Billy Madison (1995),5.0,Comedy
10,Desperado (1995),5.0,Action|Romance|Western
9,Canadian Bacon (1995),5.0,Comedy|War
8,Rob Roy (1995),5.0,Action|Drama|Romance|War
35,Pinocchio (1940),5.0,Animation|Children|Fantasy|Musical
31,Tombstone (1993),5.0,Action|Drama|Western


The catalog contains 9,742 movies; 9,724 have ratings which means there are 18 Movies with no rating.

There are 100,836 ratings from 610 users.

The checks found no missing values, repeated movie IDs, duplicate rating rows, or repeated user–movie pairs.

Ratings range from 0.5 to 5.0 in half-star increments.

Every rating matches a catalog movie.

Movies can have more than one genre label seperated by a pipe symbol.

Timestamp is unix timestamp seconds elapsed since 1/1/1970 00:00:00 UTC